In [1]:
import torch

if torch.cuda.is_available():
  print("Success! GPU is active and ready to process video.")
else:
  print("Uh oh. The GPU is not active. Check Step 2 again.")

Success! GPU is active and ready to process video.


In [2]:
import os
from google.colab import userdata

# 1. Inject your Kaggle credentials directly into the environment
os.environ['KAGGLE_USERNAME'] = 'syedabdulnafay'
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

# 2. Install the Kaggle library quietly (-q)
!pip install -q kaggle

# 3. Download the SCVD dataset directly from Kaggle's servers
print('Downloading SCVD dataset...')
!kaggle datasets download -d toluwaniaremu/smartcity-cctv-violence-detection-dataset-scvd

# 4. Unzip the downloaded file into a new folder called 'scvd_dataset'
print('Unzipping dataset... this might take a minute...')
!unzip -q smartcity-cctv-violence-detection-dataset-scvd.zip -d scvd_dataset

print('Success! Data acquired.')

Dataset URL: https://www.kaggle.com/datasets/toluwaniaremu/smartcity-cctv-violence-detection-dataset-scvd
License(s): CC-BY-NC-SA-4.0
100% 1.03G/1.03G [00:55<00:00, 19.9MB/s]

Unzipping dataset... this might take a minute...
Success! Data acquired.


In [ ]:
import os
import cv2
import numpy as np
import random
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import pandas as pd

try:
    from frame_prep import prepare_for_model
except ImportError:
    raise ImportError(
        "frame_prep.py not found in this Colab session. Upload it (same file "
        "used by camera.py in production) to the working directory before "
        "running this cell -- training now uses the same crop/prep pipeline "
        "as the live system, for the reasons documented below."
    )

# SANITY CHECK -- fail loudly on a stale frame_prep.py. The production
# frame_prep.prepare_for_model used to return 4 values (rgb, meta, gray,
# person_box); it now returns 3 (rgb, meta, gray) with the person box moved
# into meta["person_box"]. This notebook unpacks 3, so an old copy of
# frame_prep.py uploaded to Colab would otherwise die with a confusing
# "too many values to unpack" deep inside a DataLoader worker. Check once,
# up front, on a dummy frame instead. run_person_detector=False is passed
# everywhere in this notebook (except the HOG audit cell, on purpose).
_dummy_out = prepare_for_model(np.zeros((240, 320, 3), np.uint8), None, run_person_detector=False)
assert len(_dummy_out) == 3, (
    f"prepare_for_model returned {len(_dummy_out)} values, expected 3 "
    "(rgb, meta, gray). Re-upload the updated frame_prep.py to Colab."
)
assert "person_box" in _dummy_out[1], "frame_prep.py is stale: meta has no 'person_box'"
del _dummy_out

# REPRODUCIBILITY. Seed python/numpy/torch once, and give every DataLoader a
# seeded torch.Generator (controls shuffle order) plus a worker_init_fn
# (controls the per-worker `random` / numpy state used by temporal jitter
# and the horizontal flip). Without the worker seeding, each worker can
# inherit an identical copy of the parent's `random` state and produce
# correlated augmentations.
SEED = 42

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def seed_worker(worker_id):
    # torch.initial_seed() inside a worker is already base_seed + worker_id,
    # where base_seed is drawn from the loader's generator -- so it's
    # deterministic per run AND different each epoch.
    s = torch.initial_seed() % (2 ** 32)
    random.seed(s)
    np.random.seed(s)

set_seed(SEED)

# Colab gives us 2 CPU cores, and the CPU-side frame prep (decode + motion
# diff + crop + resize on up to 340 raw frames per clip) is the bottleneck,
# not the GPU -- so use both cores as loader workers. pin_memory speeds the
# host->GPU copy.
NUM_WORKERS = 2

def make_loader(ds, batch_size, shuffle):
    g = torch.Generator()
    g.manual_seed(SEED)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle,
                      num_workers=NUM_WORKERS, pin_memory=True,
                      worker_init_fn=seed_worker, generator=g)

# MAX_RAW_FRAMES was 150 (~5s @30fps), guessed defensively before we had
# real numbers. The diagnostic cell below (run after this dataset is built)
# showed Violent clips have a MEDIAN raw length of 308 frames -- more than
# double that cap -- and Violent's downsampled median/max were IDENTICAL
# (75/75), meaning most Violent clips were being truncated before we ever
# read them. If the actual violent action happens after ~5s into a longer
# clip, we were never loading those frames at all -- no amount of jitter
# tuning can fix a window that's scrambling around inside footage that
# never included the violence in the first place.
# Raised to 340 to cover the observed max (Normal max=283, Violent max=330)
# with a small margin. Re-run the diagnostic cell if the dataset changes --
# if ds median stops equalling ds max, this is no longer truncating.
MAX_RAW_FRAMES = 340

# SMART CROP INTEGRATION: two diagnostic cells below (clip-length audit,
# then a frame_prep.prepare_for_model sample audit) established that:
#   - person_crop fired 0/90 times across two independent random samples --
#     robust, not sampling noise. The default HOG pedestrian detector
#     doesn't work on this footage's camera angle/poses. Running it during
#     training would only add its real compute cost for zero benefit, so
#     it's left permanently OFF here (run_person_detector=False everywhere).
#   - motion_crop fires on ~45-55% of frames and, when it does, produces
#     reasonably-sized subject boxes (median subject_area_frac 0.055-0.095,
#     at/above frame_prep's own MIN_SUBJECT_AREA_FRAC=0.05 bar).
#   - Even in the ~50% of frames where NOTHING is detected, production
#     still applies a mild center-crop (70% of the shorter side) before
#     resizing to 112x112. Training previously applied NO crop at all --
#     a raw full-frame resize -- in every single case. That gap exists
#     regardless of whether detection succeeds, so it's worth closing
#     unconditionally, not just for the ~50% "detected" case.
# Net effect: training now runs every raw frame through the exact same
# prepare_for_model() function the live cameras use (with HOG disabled),
# instead of a bare resize -- closing a real train/inference framing gap
# using only cheap operations (motion diffing, no model inference).
USE_SMART_CROP = True


# --- VALIDATION HOLD-OUT (leak-free) ---
# There is no validation set in the manifest (only train/test), and the
# last run's train loss fell to ~0.07 by epoch 5 -- so "the saved weights"
# were just "whatever epoch 5 produced", possibly overfit. We carve ~15% of
# the TRAIN rows out as validation and never train on them. The test split
# stays untouched for the final report.
#
# Near-duplicate clips (the same source video cut into several segments)
# must land on the SAME side of the split, otherwise validation scores are
# inflated by leakage. So the split is done at the cluster level:
#   1. If the manifest has a column whose name contains 'cluster' or
#      'group', hash that value (md5) to decide train vs val.
#   2. Otherwise fall back to hashing the clip path (per-clip, not
#      per-cluster -- still deterministic, but weaker against near-duplicates).
# Hashing (rather than random sampling) means the split never changes
# between runs or Colab restarts.
import hashlib

def find_cluster_column(df):
    """Return the first manifest column whose name contains 'cluster' or 'group', else None."""
    for col in df.columns:
        low = str(col).lower()
        if 'cluster' in low or 'group' in low:
            return col
    return None

def _md5_bucket(key, buckets=10000):
    return int(hashlib.md5(str(key).encode('utf-8')).hexdigest(), 16) % buckets

def make_train_val_split(manifest_path, val_frac=0.15):
    df = pd.read_csv(manifest_path)
    df_train = df[df['split'] == 'train'].copy()
    col = find_cluster_column(df_train)
    if col is not None:
        keys = df_train[col].where(df_train[col].notna(), df_train['path']).astype(str)
        approach = f"cluster column '{col}' ({keys.nunique()} distinct groups)"
    else:
        keys = df_train['path'].astype(str)
        approach = "md5 hash of clip path (no cluster/group column found in manifest)"
    is_val = keys.map(_md5_bucket) < int(val_frac * 10000)
    print(f"Validation split approach: {approach}")
    return df_train[~is_val].copy(), df_train[is_val].copy(), approach

# --- DATASET MODES ---
#   'train' : (default when augment=True) one 16-frame window at a random
#             start (full-range temporal jitter), optional horizontal flip.
#   'first' : (default when augment=False) deterministic, frames [0:16] of
#             the stride-2 clip -- the same eval the earlier runs used, so
#             numbers stay comparable (previous run: 0.84 accuracy).
#   'multi' : eval only. Returns up to MAX_WINDOWS windows evenly spaced
#             over the whole clip as one stacked tensor, because production
#             slides a 16-frame window across the clip (main.py) and alerts
#             on a spike or a rolling average; scoring only the first ~1s
#             of a ~10s clip can "miss" clips whose opening is just calm.
#             Returns (windows[K,3,16,112,112], label, n_valid). K is padded
#             to MAX_WINDOWS by repeating the last window so the default
#             collate can batch it; use n_valid to ignore the padding.
MAX_WINDOWS = 8
WINDOW_LEN = 16

class SchoolBullyingDataset(Dataset):
  def __init__(self, manifest_path, split_type='train', augment=True, mode=None, df=None):
    """
    Args:
        manifest_path: Path to 'clean_split_manifest.csv'
        split_type: 'train' or 'test' (ignored for row selection if df is given)
        augment: Whether to apply temporal jitter + horizontal flip (train only)
        mode: 'train' | 'first' | 'multi' (see the mode notes above).
              Default: 'train' if augment else 'first'.
        df: optional pre-filtered manifest DataFrame (used for the
            train / validation hold-out split)
    """
    self.video_paths = []
    self.labels = []
    self.classes = []  # raw class names (Normal/Violence/Weaponized) for per-source recall
    # Temporal jitter and horizontal flip (see __getitem__) only apply when
    # augment=True. Keep this False for eval/test datasets so accuracy
    # numbers stay deterministic and comparable run-to-run.
    self.augment = augment
    self.mode = mode if mode is not None else ('train' if augment else 'first')
    assert self.mode in ('train', 'first', 'multi'), f"bad mode {self.mode}"

    # MERGED BACK TO 2 CLASSES, deliberately, after the 3-class run showed
    # the dominant confusion was Violent<->Weaponized (32% of real Violent
    # clips called Weaponized, 14% of real Weaponized called Violent) --
    # not danger-vs-safe. Motion alone doesn't reliably carry "was there
    # also a small handheld object" -- that's an appearance question, not
    # a temporal one. Decision: this model's only job is Danger-vs-Normal;
    # weapon-specific urgency is handled by a SEPARATE signal (the YOLO
    # weapon-presence scanner) applied only when this model already says
    # Dangerous -- see main.py integration notes. That AND-gate is also
    # what avoids false alarms on e.g. a security guard's holstered
    # sidearm: presence alone never triggers anything, only presence
    # DURING an already-flagged dangerous window does.
    #
    # NOTE: 'Weaponized' here is spelled correctly to match the real
    # folder name. Every prior run in this project's history used a typo
    # ('Weaponzied') in this exact dict, which silently matched no folder
    # on disk -- os.path.exists() returned False and the loop just skipped
    # it with no error. That means every previous "Violent" class, going
    # all the way back to the original 99% baseline, only ever contained
    # Violence clips -- the 832 (sec_split) / 100 (SCVD_converted)
    # Weaponized clips were never loaded by any training run before this
    # notebook. Historical numbers described as "Violence+Weaponized
    # merged" were not actually merged -- keep that in mind if comparing
    # against old results.

    # Load the manifest (or use the pre-split rows we were handed)
    if df is None:
        df = pd.read_csv(manifest_path)
        df_split = df[df['split'] == split_type].copy()
    else:
        df_split = df

    # Mapping: Normal -> 0, Violence/Weaponized -> 1
    class_mapping = {'Normal': 0, 'Violence': 1, 'Weaponized': 1}

    for _, row in df_split.iterrows():
        self.video_paths.append(row['path'])
        self.labels.append(class_mapping[row['class']])
        self.classes.append(row['class'])

    print(f"Loaded {len(self.video_paths)} videos for {split_type}! (augment={self.augment}, mode={self.mode})")

  def __len__(self):
    return len(self.video_paths)

  def _read_frames(self, video_path):
    # READ THE VIDEO (Using OpenCV)
    # Kept in BGR (NOT converted to RGB here) -- prepare_for_model expects
    # BGR input and does its own BGR->RGB conversion internally as part of
    # its crop+resize pipeline. Converting early would double-convert.
    cap = cv2.VideoCapture(video_path)
    frames_bgr = []
    while len(frames_bgr) < MAX_RAW_FRAMES:
      ret, frame = cap.read()
      if not ret:
        break
      frames_bgr.append(frame)
    cap.release()

    # STANDARDIZE THE TIME (Uniform Sampling - Stride 2)
    # Grab every 2nd frame from the raw video, same as before.
    return frames_bgr[::2]

  def _prep_window(self, frames_bgr, start):
    """Run one 16-frame window starting at `start` through frame_prep.

    prev_gray starts fresh for EVERY window and is seeded with the frame
    immediately preceding the window (when there is one), exactly like a
    live camera that has been running before the window began. The very
    first frame of a clip (start==0, no lookback) falls back to
    center_crop for that one frame, same as at the true start of a stream.
    """
    total = len(frames_bgr)
    lookback_frame = frames_bgr[start - 1] if start > 0 else None
    if total >= WINDOW_LEN:
      window = frames_bgr[start:start + WINDOW_LEN]
    else:
      # If the video was too short, duplicate the last frame to fill the gap
      pad_needed = WINDOW_LEN - total
      window = frames_bgr + [frames_bgr[-1]] * pad_needed if total > 0 else [np.zeros((112, 112, 3), dtype=np.uint8)] * WINDOW_LEN

    # SMART CROP + RESIZE + COLOR CONVERT (all in one pass via frame_prep)
    # Runs the SAME function the live cameras use, with HOG disabled (see
    # USE_SMART_CROP note above -- confirmed non-functional on this dataset,
    # not worth the compute).
    if USE_SMART_CROP:
      prev_gray = None
      if lookback_frame is not None:
        _, _, prev_gray = prepare_for_model(
            lookback_frame, prev_gray, run_person_detector=False
        )
      processed = []
      for f in window:
        rgb112, meta, prev_gray = prepare_for_model(
            f, prev_gray, run_person_detector=False
        )
        processed.append(rgb112)
      return np.stack(processed)  # (16, 112, 112, 3) uint8 RGB
    # Fallback path (USE_SMART_CROP=False): old behavior, bare resize, no
    # crop -- kept for A/B comparison against the smart-crop version.
    resized = [cv2.resize(cv2.cvtColor(f, cv2.COLOR_BGR2RGB), (112, 112)) for f in window]
    return np.stack(resized)

  @staticmethod
  def _to_tensor(arr):
    # TENSOR-IFY + REARRANGE FOR THE MODEL
    vframes = torch.from_numpy(np.ascontiguousarray(arr)).float() / 255.0  # (T, H, W, C)
    return vframes.permute(3, 0, 1, 2)  # (C, T, H, W) -- what the 3D model wants

  @staticmethod
  def _multi_starts(total):
    # Evenly spaced window starts over the stride-2 frames, at most
    # MAX_WINDOWS. A clip with only one possible window gets exactly one.
    if total <= WINDOW_LEN:
      return [0]
    k = min(MAX_WINDOWS, total - WINDOW_LEN + 1)
    return list(dict.fromkeys(int(s) for s in np.linspace(0, total - WINDOW_LEN, k).round()))

  def __getitem__(self, idx):
    video_path = self.video_paths[idx]
    label = torch.tensor(self.labels[idx], dtype=torch.long)
    frames_bgr = self._read_frames(video_path)
    total = len(frames_bgr)

    if self.mode == 'multi':
      starts = self._multi_starts(total)
      wins = [self._to_tensor(self._prep_window(frames_bgr, s)) for s in starts]
      n_valid = len(wins)
      while len(wins) < MAX_WINDOWS:
        wins.append(wins[-1])  # padding only; ignored via n_valid
      return torch.stack(wins), label, n_valid

    # --- TEMPORAL JITTER (full-range) ---
    # We tried bounding this to a small wobble around the clip's temporal
    # center, on the theory that a wide random window would too often land
    # on non-violent lead-in/aftermath in a long Violent-labeled clip.
    # Tested head-to-head against full-range jitter (same weighting, same
    # MAX_RAW_FRAMES=340, same layer4-unfrozen setup): bounded jitter came
    # out WORSE on the metric that matters most here -- recall dropped from
    # 86% to 73% for a precision gain of only 2 points. The likely reason:
    # violent action in these clips isn't reliably centered, so anchoring
    # near the middle can systematically exclude the real event for clips
    # where it happens early or late -- guaranteeing the model never sees
    # it for those clips, which is worse than a wide random window that at
    # least has a chance of catching it every clip. Reverted to full-range.
    if self.mode == 'train' and self.augment and total > WINDOW_LEN:
      start = random.randint(0, total - WINDOW_LEN)
    else:
      start = 0  # deterministic: eval/test, or nothing to jitter over
    arr = self._prep_window(frames_bgr, start)

    # --- HORIZONTAL FLIP (train only) ---
    # Mirroring doesn't change whether a clip is dangerous, costs almost
    # nothing, and is the standard first augmentation for action
    # recognition -- aimed at overfitting on only ~2.7k clips. p=0.5, and
    # the SAME flip is applied to every frame of the clip (flipping frames
    # independently would invent motion that never happened).
    if self.augment and random.random() < 0.5:
      arr = arr[:, :, ::-1, :]  # (T, H, W, C): flip the W axis

    return self._to_tensor(arr), label


In [ ]:
# Path to the manifest you generated with rebuild_clean_split.py
manifest_file = 'clean_split_manifest.csv'

# Hold ~15% of the TRAIN rows out as a validation set (cluster-level if the
# manifest has a cluster/group column, else a path-hash fallback -- the
# approach used is printed below). The test split is left alone.
train_df, val_df, split_approach = make_train_val_split(manifest_file, val_frac=0.15)

# Create the leakage-free training set (jitter + horizontal flip) and the
# validation set (deterministic first window, no augmentation).
my_training_data = SchoolBullyingDataset(manifest_file, split_type='train', augment=True, mode='train', df=train_df)
val_data = SchoolBullyingDataset(manifest_file, split_type='train', augment=False, mode='first', df=val_df)
train_loader = make_loader(my_training_data, batch_size=8, shuffle=True)

print(f"Train clips: {len(my_training_data)} | Validation clips: {len(val_data)}")
for name, ds in [('train', my_training_data), ('val', val_data)]:
    print(f"  {name}: " + ", ".join(f"{c}={ds.classes.count(c)}" for c in ['Normal', 'Violence', 'Weaponized']))

# Let's pull out the very first video (Index 0) and see what it looks like in math form
import time
t0 = time.time()
first_video_tensor, first_video_label = my_training_data[0]
dt = time.time() - t0

print('---')
print(f'Label: {first_video_label} (0=Normal, 1=Violent)')
print(f'Tensor Shape: {first_video_tensor.shape}')
print(f'__getitem__ took {dt:.2f}s (HOG is off; this should be much faster than with the person detector on)')
print('If the shape says [3, 16, 112, 112], your data pipeline is absolutely perfect!')


In [ ]:
# --- DIAGNOSTIC: how much room does temporal jitter actually have? ---
# Fast metadata-only scan (CAP_PROP_FRAME_COUNT reads the header, doesn't
# decode frames) across the manifest's TRAIN rows, grouped by the manifest's
# `class` column (this replaces an older version that walked a hard-coded
# train_folder_path with a 3-class label mapping). This directly measures the
# thing we've been inferring indirectly across 4 training runs: how long
# are these clips, and how much of a 16-frame training window could land
# entirely outside wherever the actual violent action happens?
import os
import cv2
import numpy as np
import pandas as pd

def get_frame_count(path):
    cap = cv2.VideoCapture(path)
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    return n

_manifest = pd.read_csv('clean_split_manifest.csv')
_train_rows = _manifest[_manifest['split'] == 'train']

class_names = ['Normal', 'Violence', 'Weaponized']
raw_counts = {name: [] for name in class_names}

for class_name, group in _train_rows.groupby('class'):
    if class_name not in raw_counts:
        raw_counts[class_name] = []
    for path in group['path']:
        if os.path.exists(path):
            raw_counts[class_name].append(get_frame_count(path))

print(f"{'Class':<12} {'n_clips':>8} {'raw min':>9} {'raw med':>9} {'raw max':>9} "
      f"{'ds min':>8} {'ds med':>8} {'ds max':>8} {'jitter room (med)':>18}")
for name, counts in raw_counts.items():
    if not counts:
        print(f"{name:<12} {0:>8}  (no readable clips)")
        continue
    arr = np.array(counts)
    capped = np.minimum(arr, MAX_RAW_FRAMES)  # __getitem__ never reads past this
    downsampled = capped // 2                  # stride-2
    jitter_room = np.clip(downsampled - 16, 0, None)  # max_start per clip
    print(f"{name:<12} {len(arr):>8} {arr.min():>9} {int(np.median(arr)):>9} {arr.max():>9} "
          f"{downsampled.min():>8} {int(np.median(downsampled)):>8} {downsampled.max():>8} "
          f"{int(np.median(jitter_room)):>18}")

print()
print("Read this as: 'jitter room' is roughly how many different starting points")
print("(in stride-2 frame units) a random 16-frame training window could be drawn")
print("from for a typical clip in that class. 0 means jitter never moves the")
print("window at all for that clip. Large values (especially for Violent) mean a")
print("training crop could plausibly land entirely inside a non-violent portion")
print("of a clip that's still labeled Violent -- the label-noise risk we've been")
print("hypothesizing about. If Violent's jitter room is small, that hypothesis")
print("is probably wrong and the remaining gap vs. the original baseline is")
print("coming from somewhere else (e.g. resolution/compression/framing")
print("differences between SCVD_converted and SCVD_converted_sec_split).")


In [ ]:
# --- DIAGNOSTIC: does the live-pipeline smart-crop actually matter for
# THIS dataset? ---
# Production (frame_prep.prepare_for_model, used in camera.py/main.py)
# detects the person/motion in a raw frame and crops+zooms to them BEFORE
# resizing to 112x112, so subjects stay a meaningful fraction of the model
# input even from a wide 1080p hallway shot. Training here just resizes the
# raw SCVD frame directly -- no cropping. If SCVD clips are already
# close/medium shots, this mismatch barely matters. If they're wide shots,
# it's a real train/inference distribution gap, same category of bug as
# the normalization mismatch we found earlier.
#
# This cell measures it directly instead of guessing: samples a few short
# bursts of frames from a subset of clips per class, runs them through the
# ACTUAL production cropping function, and reports what fraction of the
# frame the subject ends up occupying, and how often frame_prep would even
# call the scene "suitable_for_model" as-is.
#
# REQUIRES frame_prep.py to be present in this Colab session (it's not
# part of the training notebook by default -- upload it alongside your
# dataset, e.g. via the Files pane or `files.upload()`, before running this).
import random as _random
import numpy as _np

try:
    from frame_prep import prepare_for_model
except ImportError:
    raise ImportError(
        "frame_prep.py not found in this Colab session. Upload it (same file "
        "used by camera.py in production) to the working directory, then "
        "re-run this cell."
    )

SAMPLE_CLIPS_PER_CLASS = 15   # keep this small -- HOG is the slow part
BURST_LEN = 5                 # consecutive raw frames per sample point
ANCHOR_FRACTIONS = [0.25, 0.5, 0.75]  # sample near start/middle/end of each clip

def sample_scene_stats(video_paths, n_clips):
    paths = video_paths if len(video_paths) <= n_clips else _random.sample(video_paths, n_clips)
    methods = []
    subject_fracs = []
    suitable_flags = []
    for path in paths:
        cap = cv2.VideoCapture(path)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if total_frames < BURST_LEN + 1:
            cap.release()
            continue
        for frac in ANCHOR_FRACTIONS:
            start_idx = max(0, min(total_frames - BURST_LEN, int(total_frames * frac)))
            cap.set(cv2.CAP_PROP_POS_FRAMES, start_idx)
            prev_gray = None
            for _ in range(BURST_LEN):
                ok, frame = cap.read()
                if not ok:
                    break
                # HOG is deliberately LEFT ON here (default
                # run_person_detector=True): this audit is what measures
                # whether person_crop ever fires. Training cells turn it off.
                rgb112, meta, prev_gray = prepare_for_model(
                    frame, prev_gray
                )
            else:
                # only record the LAST frame of a completed burst -- it's the
                # one with valid prev_gray, so motion_crop gets a fair chance too
                methods.append(meta['method'])
                subject_fracs.append(meta['subject_area_frac'])
                suitable_flags.append(meta['suitable_for_model'])
        cap.release()
    return methods, subject_fracs, suitable_flags

print("Scanning a sample of raw SCVD_converted frames through the production "
      "cropping pipeline (frame_prep.prepare_for_model)...")
print("This can take a minute or two -- HOG detection is the slow part.\n")

from collections import Counter

for class_name, label_name in [('Normal', 'Normal'), ('Violence', 'Violence'), ('Weaponized', 'Weaponized')]:
    # Filter by the manifest's raw class NAME. (The old filter compared the
    # merged 0/1 label against 2, which no longer exists, so Weaponized was
    # never sampled.)
    paths = [p for p, c in zip(my_training_data.video_paths, my_training_data.classes)
             if c == class_name]
    if not paths:
        continue
    methods, fracs, suitable = sample_scene_stats(paths, SAMPLE_CLIPS_PER_CLASS)
    if not methods:
        print(f"{label_name}: no valid samples collected")
        continue
    method_counts = Counter(methods)
    pct_suitable = 100 * sum(suitable) / len(suitable)

    # IMPORTANT: frame_prep hardcodes subject_area_frac=0.0 whenever it falls
    # back to center_crop (no person or motion detected) -- that 0.0 is a
    # placeholder, not a measurement. Blending it into an aggregate min/
    # median/max makes "no detection happened" look like "detected a tiny
    # subject", which are very different findings. Split them apart:
    detected_fracs = _np.array([f for m, f in zip(methods, fracs) if m != 'center_crop'])
    n_no_detection = method_counts.get('center_crop', 0) + method_counts.get('full_frame', 0)
    pct_no_detection = 100 * n_no_detection / len(methods)

    print(f"--- {label_name} ({len(methods)} sampled frames from up to {SAMPLE_CLIPS_PER_CLASS} clips) ---")
    print(f"  Crop method breakdown: {dict(method_counts)}")
    print(f"  No detection at all (center_crop/full_frame fallback): {pct_no_detection:.1f}% of samples")
    if len(detected_fracs) > 0:
        print(f"  subject_area_frac WHEN SOMETHING WAS DETECTED (n={len(detected_fracs)}): "
              f"min={detected_fracs.min():.3f} median={_np.median(detected_fracs):.3f} max={detected_fracs.max():.3f}")
    else:
        print("  subject_area_frac WHEN SOMETHING WAS DETECTED: n/a (nothing was ever detected)")
    print(f"  suitable_for_model (per frame_prep's own threshold, all samples): {pct_suitable:.1f}%")
    print()

print("How to read this:")
print("- 'No detection at all' % is its own finding, separate from subject")
print("  size: it tells you how often frame_prep found NOTHING to crop to,")
print("  in which case it silently falls back to a plain center crop. High")
print("  numbers here (for either class) mean a large chunk of frames never")
print("  even reach the crop-quality question -- they're centered-and-hoped,")
print("  not smart-cropped.")
print("- 'subject_area_frac WHEN SOMETHING WAS DETECTED' is the honest read")
print("  on subject scale, restricted to frames where a real box existed.")
print("  Compare its median to MIN_SUBJECT_AREA_FRAC=0.05.")
print("- If person_crop count is 0 or near-0 across both classes: the HOG")
print("  pedestrian detector (trained on upright, street-level walking")
print("  people) likely doesn't fire reliably on this footage's camera")
print("  angle/poses. That is a PRODUCTION-relevant finding on its own,")
print("  independent of the training question -- if live cameras share a")
print("  similar mounting angle, person_crop may rarely fire there either,")
print("  meaning the live system is leaning almost entirely on the motion")
print("  fallback (and plain center_crop when even that fails) in practice.")
print("  Worth checking against real camera footage separately.")


In [7]:
import torch
import torch.nn as nn
import torchvision.models.video as video_models

print("Downloading the pre-trained R3D_18 model (This might take a few seconds)...")

# 1. LOAD THE PRE-TRAINED MODEL
# We pull the model and its pre-learned Kinetics-400 weights
weights = video_models.R3D_18_Weights.DEFAULT
model = video_models.r3d_18(weights=weights)

# 2. FREEZE THE KNOWLEDGE (Transfer Learning)
# We loop through all the existing "neurons" and tell PyTorch NOT to update them.
# This saves massive computing power and prevents the model from forgetting the basics.
for param in model.parameters():
  param.requires_grad = False

# 2.5 PARTIAL UNFREEZE (layer4) -- TOGGLE FOR ABLATION
# r3d_18 is stem -> layer1 -> layer2 -> layer3 -> layer4 -> avgpool -> fc.
# layer1-3 learn low-level stuff (edges, textures, generic motion primitives)
# that's the same whether the action is a handshake or a punch -- no reason
# to retrain those. layer4 is the deepest residual block group (it contains
# 2 residual blocks) and is where the highest-level spatiotemporal features
# live -- exactly the level where "violent contact" vs "generic fast human
# motion" needs to be distinguished.
#
# UNFREEZE_LAYER4 = False reproduces the original fc-only setup (on top of
# whatever dataset/jitter/weighting is currently configured elsewhere in
# the notebook). Run it both ways with everything else held constant to
# see how much of the precision/recall shift across your recent runs is
# actually attributable to this layer, versus the dataset swap + jitter.
UNFREEZE_LAYER4 = True  # confirmed better than False: see Run 3 vs Run 4 comparison

if UNFREEZE_LAYER4:
  for param in model.layer4.parameters():
    param.requires_grad = True
  print("layer4: UNFROZEN (trainable)")
else:
  print("layer4: frozen (fc-only training, like the original baseline)")

# 3. THE SURGERY (Replacing the Head)
# We find out how many connections are flowing into the final layer
num_features = model.fc.in_features

# We replace the old 400-class layer with a new 2-class layer (0: Normal, 1: Violent)
model.fc = nn.Linear(num_features, 2)

# We make sure the this brand new layer is unlocked and ready to learn
for param in model.fc.parameters():
  param.requires_grad = True

# Sanity check: confirm what's actually trainable before you spend GPU time
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)")
print("Expect this to include layer4.* and fc.* only -- if it's just fc, the unfreeze above didn't take.")

# 4. MOVE TO THE SUPERCOMPUTER
# We tell PyTorch to grab our newly modified model and throw it onto the GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print("\n--- Surgery Complete ---")
print(f"Model successfully moved to: {device}")
print("If the device above says 'cuda', your GPU is active and the brain is ready!")

Downloading: "https://download.pytorch.org/models/r3d_18-b3b3357e.pth" to /root/.cache/torch/hub/checkpoints/r3d_18-b3b3357e.pth


100%|██████████| 127M/127M [00:00<00:00, 186MB/s]


layer4: UNFROZEN (trainable)
Trainable params: 24,909,826 / 33,167,298 (75.1%)
Expect this to include layer4.* and fc.* only -- if it's just fc, the unfreeze above didn't take.

--- Surgery Complete ---
Model successfully moved to: cuda
If the device above says 'cuda', your GPU is active and the brain is ready!


In [ ]:
# DRIVE CHECKPOINT: a closed laptop lid / browser disconnect can kill the
# Colab session and wipe in-memory weights. Mount Drive up front so the best
# epoch is written to MyDrive/safespace_checkpoints/ every time it improves.
# Recover later with: ckpt = torch.load(path); model.load_state_dict(ckpt["state_dict"])
import os
CHECKPOINT_DIR = None
try:
  from google.colab import drive
  drive.mount("/content/drive")
  CHECKPOINT_DIR = "/content/drive/MyDrive/safespace_checkpoints"
  os.makedirs(CHECKPOINT_DIR, exist_ok=True)
  print(f"Best-epoch checkpoints will be saved to {CHECKPOINT_DIR}")
except Exception as e:
  print(f"WARNING: Google Drive not mounted ({e}) -- best weights will only live in memory.")

import copy
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score

print("Setting up the classroom...")

# 1. BATCHING (The DataLoader)
# Video tensors are massive. If we try to feed all our videos to the GPU at once,
# it will instantly crash (Out of Memory error). We group them into tiny batches of 8.
batch_size = 8
# make_loader (defined in the dataset cell) = num_workers=2, pin_memory=True,
# seeded worker_init_fn and a seeded torch.Generator for the shuffle order.
train_loader = make_loader(my_training_data, batch_size=batch_size, shuffle=True)
val_loader = make_loader(val_data, batch_size=batch_size, shuffle=False)

# 2. THE GRADER & THE STUDENT
# CrossEntropyLoss is the standard mathematical formula for grading multiple-choice questions
#
# CLASS WEIGHTING -- TOGGLE FOR ABLATION.
# We added this to fight a recall collapse (65%) that turned out to be
# caused mostly by a separate bug: MAX_RAW_FRAMES was truncating most
# Violent clips before their actual violent content was ever read, so the
# model was structurally starved of real signal. Weighting masked that by
# forcing more "guess Violent" bias, at the cost of precision dropping to
# ~75% and staying there regardless of weight strength (2.02x and 1.42x
# gave nearly identical results) -- a sign it wasn't the right lever.
# Now that MAX_RAW_FRAMES=340 actually fixes the truncation, it's worth
# testing whether weighting is still needed at all, or whether the model
# recovers recall on its own now that it can see genuine violent footage --
# and whether precision comes back toward the original 100% once we stop
# artificially biasing the decision boundary.
# Run 6 (unweighted, full-range jitter, fixed truncation) settled this:
# recall collapsed to 55% without weighting -- WORSE than even the old
# truncated run. The 2:1 class imbalance is real and weighting is needed
# independent of the truncation bug. Re-enabled here, now paired with the
# bounded/center-anchored jitter above, which targets the label-noise
# problem that weighting alone couldn't fully compensate for.
#
# Run 7 update -- THE IMBALANCE HAS FLIPPED. All of the history above was
# written when Violent was the minority class. After the Weaponized clips
# were (correctly) loaded, the training set is Normal ~1029 vs Dangerous
# ~1653 (before the validation hold-out), so Normal is now the minority and
# Normal recall -- i.e. FALSE ALARMS -- is the weak metric: 0.67 on the last
# run, about 1 in 3 normal clips flagged, versus Dangerous recall 0.90.
# Turning the already-computed softened (sqrt) weights ON therefore pushes
# in the right direction this time: roughly [1.14, 0.90] (Normal up,
# Dangerous down). sqrt rather than full inverse-frequency, because the
# earlier runs showed full-strength weighting just trades one error for the
# other and wrecks precision. Judge it on Normal recall in the report.
USE_CLASS_WEIGHTS = True

label_counts = torch.bincount(torch.tensor(my_training_data.labels))
num_classes = len(label_counts)
num_samples = len(my_training_data.labels)
raw_weights = num_samples / (num_classes * label_counts.float())
sqrt_weights = raw_weights.sqrt()
print(f"Class counts (Normal, Dangerous) in the TRAIN subset: {label_counts.tolist()}")
print(f"Raw inverse-frequency weights (unused unless noted): {raw_weights.tolist()}")
print(f"Softened (sqrt) weights (unused unless noted): {sqrt_weights.tolist()}")

if USE_CLASS_WEIGHTS:
    class_weights = sqrt_weights.to(device)
    print(f"USE_CLASS_WEIGHTS=True -> training with weights: {class_weights.tolist()}")
    criterion = nn.CrossEntropyLoss(weight=class_weights)
else:
    print("USE_CLASS_WEIGHTS=False -> plain unweighted CrossEntropyLoss")
    criterion = nn.CrossEntropyLoss()

# The Optimizer is the student. It updates the weights based on the Grader's feedback.
# Two param groups now that layer4 is unfrozen too (see the surgery cell):
#   - model.fc: brand new, random-initialized layer -> normal LR (1e-3)
#   - model.layer4: pretrained Kinetics-400 weights, already good at general
#     motion -> a much smaller LR (1e-5) so we nudge them toward
#     violence-specific features instead of overwriting what they already
#     know with only a few epochs on a dataset far smaller than Kinetics-400.
# Respects the UNFREEZE_LAYER4 toggle from the surgery cell -- when False,
# layer4 params are left out of the optimizer entirely (explicit, rather
# than relying on requires_grad=False silently making Adam skip them).
param_groups = [{'params': model.fc.parameters(), 'lr': 0.001}]
if UNFREEZE_LAYER4:
    param_groups.append({'params': model.layer4.parameters(), 'lr': 0.00001})
optimizer = optim.Adam(param_groups)

# 3. THE LOOP
# An "Epoch" is one complete read-through of the entire training textbook (dataset)
# Raised from 5 to 8: train loss was already ~0.07 by epoch 5 last time, and
# with per-epoch validation + best-checkpoint keeping, extra epochs are
# cheap insurance -- we keep the BEST epoch, not the last one.
epochs = 8

# PER-EPOCH VALIDATION (first-window, fast): same deterministic eval as the
# earlier runs, on the held-out slice of train. Macro-F1 (Normal and
# Dangerous weighted equally) is the selection metric, since plain accuracy
# would reward ignoring the minority class.
def evaluate_first_window(model, loader):
  model.eval()
  probs, labs = [], []
  with torch.no_grad():
    for inputs, labels in loader:
      p = F.softmax(model(inputs.to(device)), dim=1)[:, 1]
      probs.extend(p.cpu().numpy())
      labs.extend(labels.numpy())
  probs, labs = np.array(probs), np.array(labs)
  pred = (probs >= 0.5).astype(int)
  return {
      'macro_f1': float(f1_score(labs, pred, average='macro')),
      'accuracy': float((pred == labs).mean()),
      'normal_recall': float((pred[labs == 0] == 0).mean()) if (labs == 0).any() else float('nan'),
      'dangerous_recall': float((pred[labs == 1] == 1).mean()) if (labs == 1).any() else float('nan'),
  }

best_val_f1 = -1.0
best_epoch = 0
best_state = None          # in-memory copy of the best weights (copy.deepcopy)
best_val_metrics = None
history = []

print("Starting the Training Loop! Grab a coffee, this will take a few minutes...")

for epoch in range(epochs):
  model.train() # Set the model to "Learning Mode"
  running_loss = 0.0

  # We loop through our dataset, grabbing 8 videos (inputs) and 8 answers (labels) at a time
  for i, (inputs, labels) in enumerate(train_loader):

    # Move the data from the CPU's RAM onto the GPU's ultra-fast memory
    inputs = inputs.to(device)
    labels = labels.to(device)

    # Step A: Wipe the slate clean
    # PyTorch accumulates gradients (learning signals) by default. We must reset them per batch.
    optimizer.zero_grad()

    # Step B: The Forward Pass (Make a guess)
    outputs = model(inputs)

    # Step C: Calculate the Error (How wrong was the guess?)
    loss = criterion(outputs, labels)

    # Step D: The Backward Pass (Learn from the mistake)
    # This calculates exactly how much to adjust the math in the final layer
    loss.backward()

    # Apply the adjustments
    optimizer.step()

    running_loss += loss.item()

    # Print a progress update every 10 batches so we know it hasn't frozen
    if (i + 1) % 10 == 0:
      print(f"Epoch [{epoch+1}/{epochs}], Batch [{i+1}/{len(train_loader)}], Loss: {loss.item():.4f}")

  # Print the average error at the end of the every epoch.
  # We want to see the number go DOWN over time!
  epoch_loss = running_loss / len(train_loader)
  print(f"--- End of Epoch {epoch + 1} | Average Loss: {epoch_loss:.4f} ---")

  # Validate on the held-out slice and remember the best epoch's weights.
  val_metrics = evaluate_first_window(model, val_loader)
  history.append({'epoch': epoch + 1, 'train_loss': epoch_loss, **val_metrics})
  print(f"    Val (first-window): macro-F1={val_metrics['macro_f1']:.4f} "
        f"acc={val_metrics['accuracy']:.4f} "
        f"Normal recall={val_metrics['normal_recall']:.3f} "
        f"Dangerous recall={val_metrics['dangerous_recall']:.3f}")
  if val_metrics['macro_f1'] > best_val_f1:
    best_val_f1 = val_metrics['macro_f1']
    best_epoch = epoch + 1
    best_val_metrics = val_metrics
    best_state = copy.deepcopy(model.state_dict())
    print(f"    ** new best (epoch {best_epoch}) -- weights kept in memory **")
    if CHECKPOINT_DIR:
      _ckpt = os.path.join(CHECKPOINT_DIR, "safespace_v3_best_checkpoint.pth")
      torch.save({"state_dict": best_state, "epoch": best_epoch, "val_metrics": best_val_metrics}, _ckpt)
      print(f"    ** also saved to Google Drive: {_ckpt} **")

print(f"\n Training Complete! Best epoch: {best_epoch} (val macro-F1 {best_val_f1:.4f}). "
      "The evaluation cell loads these weights, not the last epoch's.")

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch.nn.functional as F
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, average_precision_score)

print("Setting up the Evaluation Engine...")

# 0. Load the BEST epoch's weights (by validation macro-F1), not whatever the
# last epoch left in the model.
model.load_state_dict(best_state)
print(f"Loaded best weights: epoch {best_epoch}, val macro-F1 {best_val_f1:.4f}")

# 1. Locate the Test Data
manifest_file = 'clean_split_manifest.csv'

# 2. Create the Dataset and DataLoader
# mode='multi': each clip comes back as up to 8 windows evenly spaced over
# the whole clip (see the dataset cell). Window 0 is exactly the old
# "first 16 stride-2 frames" eval -- identical frames and prev_gray handling
# -- so the first-window numbers below are directly comparable to the
# earlier 0.84-accuracy run WITHOUT a second inference pass.
# augment=False: eval must be deterministic (no jitter, no flip).
test_data = SchoolBullyingDataset(manifest_file, split_type='test', augment=False, mode='multi')

# shuffle=False is important here. We want to test them in a straight line.
test_batch_size = 8
test_loader = make_loader(test_data, batch_size=test_batch_size, shuffle=False)

# 3. Storage for our results (per-clip window probabilities are KEPT so the
# miss-review cell can reuse them instead of running inference again).
all_win_probs = []   # list of np arrays: P(Dangerous) for each valid window of each clip
all_true_labels = []

# 4. Lock the Brain
model.eval() # Strictly testing mode, no learning!

print(f"\nEvaluating {len(test_data)} unseen videos (up to {MAX_WINDOWS} windows each). This will take a few minutes...")

# 5. The Evaluation Loop
with torch.no_grad(): # Tells the GPU not to waste memory tracking gradients
    for i, (inputs, labels, n_valid) in enumerate(test_loader):
        # inputs: [B, K, C, T, H, W]; drop the padding windows, then run the
        # real ones through the model in chunks.
        K = inputs.shape[1]
        mask = torch.arange(K)[None, :] < n_valid[:, None]
        flat = inputs[mask]
        probs = torch.cat([
            F.softmax(model(chunk.to(device)), dim=1)[:, 1].cpu()
            for chunk in flat.split(32)
        ])
        for p in probs.split(n_valid.tolist()):
            all_win_probs.append(p.numpy())
        all_true_labels.extend(labels.numpy())

        # Print a tiny update so you know it hasn't frozen
        if (i + 1) % 10 == 0:
            print(f"Processed {min((i + 1) * test_batch_size, len(test_data))} videos...")

# 6. Per-clip scores. Three ways to turn a clip's windows into one number:
#   first = window 0 only          (old eval; comparable to earlier runs)
#   max   = highest window         (mirrors main.py's SPIKE logic)
#   mean  = average over windows   (mirrors the rolling-average ALERT logic)
y_true = np.array(all_true_labels)
results_df = pd.DataFrame({
    'path': test_data.video_paths,
    'class': test_data.classes,
    'y_true': y_true,
    'n_windows': [len(w) for w in all_win_probs],
    'first_p': [w[0] for w in all_win_probs],
    'max_p': [w.max() for w in all_win_probs],
    'mean_p': [w.mean() for w in all_win_probs],
})
scores = {'first-window': results_df['first_p'].values,
          'multi max': results_df['max_p'].values,
          'multi mean': results_df['mean_p'].values}

ALERT_THRESHOLD = 0.67   # main.py: rolling-average alert
SPIKE_THRESHOLD = 0.90   # main.py: single-window spike
THRESHOLDS = [0.5, ALERT_THRESHOLD, SPIKE_THRESHOLD]

# 7. Generate the Mathematical Report
print("\n" + "="*40)
print("     OFFICIAL PERFORMANCE REPORT")
print("="*40)
print("Previous run (first-window, argmax, last-epoch weights): accuracy 0.84, "
      "Dangerous P/R 0.88/0.90, Normal recall 0.67\n")
# 'Dangerous' covers both Violence and Weaponized clips -- see the merge
# note in the dataset cell for why they're back to one label.
target_names = ['Normal (0)', 'Dangerous (1)']
for name, s in scores.items():
    pred = (s >= 0.5).astype(int)
    print(f"##### {name} @ threshold 0.5 #####")
    print(classification_report(y_true, pred, labels=[0, 1], target_names=target_names))
    print(f"ROC-AUC: {roc_auc_score(y_true, s):.4f} | PR-AUC (avg precision, Dangerous=positive): "
          f"{average_precision_score(y_true, s):.4f}\n")

# 8. Confusion matrices at the thresholds production actually uses:
# 0.5 (plain argmax), 0.67 (ALERT_THRESHOLD), 0.90 (SPIKE_THRESHOLD).
fig, axes = plt.subplots(len(scores), len(THRESHOLDS), figsize=(5 * len(THRESHOLDS), 4 * len(scores)))
for r, (name, s) in enumerate(scores.items()):
    for c, th in enumerate(THRESHOLDS):
        pred = (s >= th).astype(int)
        cm = confusion_matrix(y_true, pred, labels=[0, 1])
        print(f"{name} @ {th:.2f}: [[TN={cm[0,0]}, FP={cm[0,1]}], [FN={cm[1,0]}, TP={cm[1,1]}]]  "
              f"Normal recall={cm[0,0]/max(cm[0].sum(),1):.3f}  Dangerous recall={cm[1,1]/max(cm[1].sum(),1):.3f}")
        sns.heatmap(cm, annot=True, fmt='d', cmap='Reds', ax=axes[r, c], cbar=False,
                    xticklabels=['Pred Normal', 'Pred Dangerous'],
                    yticklabels=['Actually Normal', 'Actually Dangerous'])
        axes[r, c].set_title(f"{name} @ {th:.2f}")
    print()
plt.suptitle('SafeSpace System: Confusion Matrices (Danger vs Normal)')
plt.tight_layout()
plt.show()

# 9. Per-source recall: is Weaponized easier or harder than Violence?
print("Dangerous recall by source class (fraction of clips scored >= threshold):")
print(f"{'score':<14} {'thr':>5} {'Violence':>10} {'Weaponized':>12}")
for name, s in scores.items():
    for th in THRESHOLDS:
        row = []
        for cls_name in ['Violence', 'Weaponized']:
            m = (results_df['class'] == cls_name).values
            row.append(f"{(s[m] >= th).mean():.3f} (n={m.sum()})" if m.any() else "n/a")
        print(f"{name:<14} {th:>5.2f} {row[0]:>10} {row[1]:>12}")


In [ ]:
# --- INSPECT MISSES: what is the model actually failing to see? ---
# Pulls every test clip where the model scored Normal but the true label
# was Dangerous (the ~97 first-window misses from the last eval run), sorted by how
# CONFIDENT the model was in that wrong answer. That distinction matters:
# a miss where the model gave Dangerous only 4% probability is a genuine
# blind spot (never learned to recognize this pattern at all). A miss at
# 47% is a coin-flip the model almost got right -- a fundamentally
# different, less concerning kind of failure, and a candidate for
# threshold tuning rather than a real model weakness.
#
# For each miss, this pulls several frames spanning the FULL raw clip (not
# just the 16 frames the model happened to see this run) so you can check
# a specific hypothesis directly: does the actual dangerous content sit
# somewhere in the clip that jitter could easily have missed, or is this
# genuinely calm-looking footage that's just mislabeled/ambiguous?
import os
import cv2
import numpy as np

MISS_OUTPUT_DIR = "miss_review"
N_CONTEXT_FRAMES = 5  # spread across the whole raw clip, not just what jitter sampled
os.makedirs(MISS_OUTPUT_DIR, exist_ok=True)

def get_context_frames(path, n=N_CONTEXT_FRAMES):
    cap = cv2.VideoCapture(path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total <= 0:
        cap.release()
        return []
    positions = [int(total * i / (n - 1)) if n > 1 else total // 2 for i in range(n)]
    positions = [min(p, total - 1) for p in positions]
    frames = []
    for pos in positions:
        cap.set(cv2.CAP_PROP_POS_FRAMES, pos)
        ok, frame = cap.read()
        if ok:
            frames.append((pos, frame))
    cap.release()
    return frames

# Reuses results_df from the evaluation cell -- NO second inference pass.
# MISS_SCORE picks which clip score defines a "miss":
#   'first_p' = first window only (the old definition, ~97 misses last run)
#   'max_p'   = best window anywhere in the clip (mirrors SPIKE)
#   'mean_p'  = average over windows (mirrors the rolling ALERT)
MISS_SCORE = 'max_p'
MISS_THRESHOLD = 0.5

missed = results_df[(results_df['y_true'] == 1) & (results_df[MISS_SCORE] < MISS_THRESHOLD)]
miss_records = [(r.path, r.first_p, r.max_p, getattr(r, MISS_SCORE)) for r in missed.itertuples()]

print(f"\n{len(miss_records)} misses found (true=Dangerous, {MISS_SCORE} < {MISS_THRESHOLD}).")

# Most-confidently-wrong first -- these are the real blind spots to look at.
miss_records.sort(key=lambda r: r[3])

print(f"Saving {len(miss_records)} review strips to {MISS_OUTPUT_DIR}/ ...")
for rank, (path, first_p, max_p, dangerous_prob) in enumerate(miss_records):
    context = get_context_frames(path)
    if not context:
        continue
    thumbs = []
    for pos, frame in context:
        thumb = cv2.resize(frame, (160, 120))
        cv2.putText(thumb, f"f{pos}", (4, 14), cv2.FONT_HERSHEY_SIMPLEX,
                    0.4, (255, 255, 255), 1, cv2.LINE_AA)
        thumbs.append(thumb)
    strip = np.hstack(thumbs)
    header = np.full((24, strip.shape[1], 3), 30, dtype=np.uint8)
    label = f"{os.path.basename(path)}  |  Dangerous prob: first={first_p:.2f} max={max_p:.2f}"
    cv2.putText(header, label, (6, 17), cv2.FONT_HERSHEY_SIMPLEX,
                0.45, (255, 255, 255), 1, cv2.LINE_AA)
    canvas = np.vstack([header, strip])

    out_name = f"{rank:03d}_prob{dangerous_prob:.2f}_{os.path.basename(path)}.jpg"
    cv2.imwrite(os.path.join(MISS_OUTPUT_DIR, out_name), canvas)

print(f"\nDone. Files are named rank_probability_filename.jpg -- sorted so the")
print(f"lowest-numbered files are the model's most confident wrong answers.")
print(f"Start there: those are the clearest signal of what the model has")
print(f"never learned to recognize, versus the higher-probability files near")
print(f"the end of the list, which were closer to a coin flip.")


In [ ]:
from google.colab import files

# 1. Define the filenames for your saved brain
# New name (v3) so you keep the earlier weights around for side-by-side
# comparison with video_calibration.py --model-path.
#   - model_filename: a PLAIN state_dict -- exactly what main.py loads.
#   - full_filename:  a dict with the state_dict plus provenance (epoch,
#     validation metrics, config flags) for the project record. main.py
#     would need a one-line change (ckpt['state_dict']) to load this one.
model_filename = 'safespace_bullying_model_v3_weaponized.pth'
full_filename = 'safespace_bullying_model_v3_weaponized_full.pth'

# 2. Save the BEST epoch's weights (by validation macro-F1), not the last.
print(f"Saving best-epoch weights (epoch {best_epoch}) to {model_filename} and {full_filename}...")
torch.save(best_state, model_filename)
torch.save({
    'state_dict': best_state,
    'epoch': best_epoch,
    'val_metrics': best_val_metrics,
    'history': history,
    'config': {
        'seed': SEED,
        'epochs': epochs,
        'USE_CLASS_WEIGHTS': USE_CLASS_WEIGHTS,
        'class_weights': (sqrt_weights.tolist() if USE_CLASS_WEIGHTS else None),
        'UNFREEZE_LAYER4': UNFREEZE_LAYER4,
        'USE_SMART_CROP': USE_SMART_CROP,
        'MAX_RAW_FRAMES': MAX_RAW_FRAMES,
        'horizontal_flip': True,
        'val_split': split_approach,
        'num_classes': 2,
        'class_names': ['Normal', 'Dangerous'],
    },
}, full_filename)

# 3. Download both files from the cloud directly to your local computer
print("Downloading to your computer... check your browser downloads!")
files.download(model_filename)
files.download(full_filename)


In [11]:
from pathlib import Path
from google.colab import files

folder_path = Path("miss_review")
output_file = Path("miss_review_files.txt")

# Collect only files (skips nested directories)
file_names = [f.name for f in folder_path.iterdir() if f.is_file()]

# Write filenames to text file
with open(output_file, "w", encoding="utf-8") as f:
    f.write("\n".join(file_names))

# Trigger browser download (Google Colab)
files.download(output_file)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>